# BCUL importer Debug for new batches

### Imports

In [1]:
# Automatically reloads modules when you make changes (useful during development)
%load_ext autoreload
%autoreload 2

In [2]:
import logging
import os
import json
import string
from collections import namedtuple
import sys
import tqdm

from dask import bag as db


In [3]:
# add the parent directory to the path
sys.path.append("/rcp-scratch/iccluster040_scratch/students/banuls/impresso-essentials")

In [4]:
from text_preparation.importers.detect import _apply_datefilter
from text_preparation.importers.bcul.helpers import parse_date, find_mit_file
from text_preparation.importers.bcul.classes import BculNewspaperIssue


logger = logging.getLogger(__name__)

BculIssueDir = namedtuple(
    "IssueDirectory", ["provider", "alias", "date", "edition", "path", "mit_file_type"]
)

In [5]:
BASE_DIR = "/mnt/project_impresso/original/BCUL"

In [6]:
OLD_ALIASES_FILEPATH = '/rcp-scratch/iccluster040_scratch/students/banuls/impresso-text-acquisition/text_preparation/data/sample_data/BCUL/bcul_aliases.json'
ALIASES_FILEPATH = '/rcp-scratch/iccluster040_scratch/students/banuls/impresso-text-acquisition/text_preparation/data/sample_data/BCUL/bcul_aliases3_4.json'


## Detect


In [7]:
def old_dir2issue(path: str, journal_info: dict[str, str]) -> BculIssueDir | None:
    """Create a `BculIssueDir` object from a directory.

    Note:
        This function is called internally by `detect_issues`

    Args:
        path (str): The path of the issue.
        access_rights (dict): Dictionary for access rights.

    Returns:
        BculIssueDir | None: New `BculIssueDir` object.
    """
    mit_file = find_mit_file(path)
    if mit_file is None:
        logger.error("Could not find MIT file in %s", path)
        return None

    mit_ext = mit_file.split(".")[-1]
    expected_ext = journal_info["file_type"]
    print('mit file ends with:', mit_file, mit_ext, expected_ext)
    # --- handle 'both' case --- 
    if expected_ext == "both":
        if mit_ext not in ('xml', 'json'):
            logger.warning(
                "Found mit file %s has unexpected extension %s, expected 'xml' or 'json'",
                os.path.join(path, mit_file),
                mit_ext,
            )
            # accept either format without changing journal_info
    else: 
        # --- normal case ---
        if not mit_file.endswith(journal_info["file_type"]):
            logger.warning(
                "Found mit file %s does not correspond to mit file type %s",
                os.path.join(path, mit_file),
                expected_ext,
            )
            # override the mit file type if the extension of the file found does not match
            journal_info["file_type"] = mit_ext

    date = parse_date(mit_file)

    # check if multiple issues are at this date:
    day_dir = os.path.dirname(path)
    day_editions = list(os.listdir(day_dir))
    day_editions = [
        str(i)
        for i in os.listdir(day_dir)
        if i != ".DS_Store"
    ]

    if len(day_editions) > 1:
        # if multiple issues exist for a given day, find the correct edition
        logger.info("Multiple issues for %s, finding the edition", day_dir)
        # exclude incorrect issues from the list
        index = sorted(day_editions).index(os.path.basename(path))
        edition = string.ascii_lowercase[index]
    else:
        edition = "a"

    return BculIssueDir(
        provider="BCUL",
        alias=journal_info["alias"],
        date=date,
        edition=edition,
        path=path,
        mit_file_type=mit_ext if expected_ext == "both" else journal_info["file_type"],
    )


In [8]:
def dir2issue(path: str, journal_info: dict[str, str]) -> BculIssueDir | None:
    """Create a `BculIssueDir` object from a directory.

    Note:
        This function is called internally by `detect_issues`

    Args:
        path (str): The path of the issue.
        access_rights (dict): Dictionary for access rights.

    Returns:
        BculIssueDir | None: New `BculIssueDir` object.
    """
    mit_file = find_mit_file(path)
    if mit_file is None:
        logger.error("Could not find MIT file in %s", path)
        return None

    mit_ext = mit_file.split(".")[-1]
    expected_ext = journal_info["mit_file_type"]
    print('mit file ends with:', mit_file, mit_ext, expected_ext)
    # --- handle 'both' case --- 
    if expected_ext == "both":
        if mit_ext not in ('xml', 'json'):
            logger.warning(
                "Found mit file %s has unexpected extension %s, expected 'xml' or 'json'",
                os.path.join(path, mit_file),
                mit_ext,
            )
            # accept either format without changing journal_info
    else: 
        # --- normal case ---
        if not mit_file.endswith(journal_info["mit_file_type"]):
            logger.warning(
                "Found mit file %s does not correspond to mit file type %s",
                os.path.join(path, mit_file),
                expected_ext,
            )
            # override the mit file type if the extension of the file found does not match
            journal_info["mit_file_type"] = mit_ext

    date = parse_date(mit_file)

    # check if multiple issues are at this date:
    day_dir = os.path.dirname(path)
    day_editions = list(os.listdir(day_dir))
    day_editions = [
        str(i)
        for i in os.listdir(day_dir)
        if i != ".DS_Store"
    ]

    if len(day_editions) > 1:
        # if multiple issues exist for a given day, find the correct edition
        logger.info("Multiple issues for %s, finding the edition", day_dir)
        # exclude incorrect issues from the list
        index = sorted(day_editions).index(os.path.basename(path))
        edition = string.ascii_lowercase[index]
    else:
        edition = "a"

    return BculIssueDir(
        provider="BCUL",
        alias=journal_info["alias"],
        date=date,
        edition=edition,
        path=path,
        mit_file_type=mit_ext if expected_ext == "both" else journal_info["mit_file_type"],
    )


## DO NOT RUN IT TAKES FOREVER

In [ ]:
# open and read bcul_alias.json file
with open(ALIASES_FILEPATH, "rb") as f:
    alias_mapping = json.load(f)

dir_path, dirs, files = next(os.walk(BASE_DIR))

journal_dirs = [
    os.path.join(dir_path, _dir)
    for _dir in dirs
    if _dir not in ["OLD", "wrong_BCUL", ".DS_Store"] and _dir in alias_mapping
]
issue_dirs = []
for journal in journal_dirs:
    logger.info("Detecting issues for %s.", journal)
    for dir_path, dirs, files in os.walk(journal):
        title = journal.split("/")[-1]
        # check if we are in the directory of a (valid) issue
        if (
            len(files) > 1
            and "solr" not in dir_path
        ):
            issue_dirs.append(dir2issue(dir_path, alias_mapping[title]))

# return issue_dirs

In [8]:
journal = "/mnt/project_impresso/original/BCUL/Domaine_Public"

In [88]:
issue_dirs = []
nb_issues = 0
for dir_path, dirs, files in os.walk(journal):
    title = journal.split("/")[-1]
    if (
            len(files) > 1
            and "solr" not in dir_path
        ):
            nb_issues += 1
            issue_dirs.append(dir2issue(dir_path, alias_mapping[title]))


    

## bcul.classes.py

In [9]:
# open and read bcul_alias.json file
with open(OLD_ALIASES_FILEPATH, "rb") as f:
    old_alias_mapping = json.load(f)

In [10]:
# open and read bcul_alias.json file
with open(ALIASES_FILEPATH, "rb") as f:
    alias_mapping = json.load(f)

In [11]:
MEdir = old_dir2issue("/mnt/project_impresso/original/BCUL/Le_Grelot/1845/09/01/127522", old_alias_mapping["Le_Grelot"])
CONFdir = dir2issue("/mnt/project_impresso/original/BCUL/Confiance/1950/00/00/394437", alias_mapping["Confiance"])
DPdir = dir2issue("/mnt/project_impresso/original/BCUL/Domaine_Public/1987/10/15/169220", alias_mapping["Domaine_Public"])

mit file ends with: /mnt/project_impresso/original/BCUL/Le_Grelot/1845/09/01/127522/Grelot_0012_1845_09_01_0001_mit.xml xml xml
mit file ends with: /mnt/project_impresso/original/BCUL/Confiance/1950/00/00/394437/EM_1950_09_00_mit.json json json
mit file ends with: /mnt/project_impresso/original/BCUL/Domaine_Public/1987/10/15/169220/DP_0879_1987_10_15_01_mit.xml xml both


In [12]:
print(DPdir)

IssueDirectory(provider='BCUL', alias='DP', date=datetime.date(1987, 10, 15), edition='a', path='/mnt/project_impresso/original/BCUL/Domaine_Public/1987/10/15/169220', mit_file_type='xml')


In [13]:
issue = BculNewspaperIssue(DPdir)
for p in issue.pages:
    print(p.page_data["id"], p.page_data.get("fw"), p.page_data.get("fh"))


DP-1987-10-15-a-p0001 2256 3015
DP-1987-10-15-a-p0002 2256 3015
DP-1987-10-15-a-p0003 2256 3015
DP-1987-10-15-a-p0004 2256 3015
DP-1987-10-15-a-p0005 2256 3015
DP-1987-10-15-a-p0006 2256 3015
DP-1987-10-15-a-p0007 2256 3015
DP-1987-10-15-a-p0008 2256 3015


In [14]:
issue.pages[0].page_data


{'id': 'DP-1987-10-15-a-p0001',
 'cdt': '2025-11-05 16:56:21',
 'ts': '2025-11-05T15:56:21Z',
 'st': 'newspaper',
 'sm': 'print',
 'r': [],
 'iiif_img_base_uri': 'https://www.scriptorium.ch/api/iiif-img/v3/1782642',
 'fw': 2256,
 'fh': 3015}

In [15]:
import requests, json
url = "https://scriptorium.bcu-lausanne.ch/api/iiif/168346/manifest"
response = requests.get(url, verify=False)
print(response.status_code)

200


In [16]:
file_path = '/mnt/project_impresso/original/BCUL/Confiance/1950/00/00/394437/6355589_exif.json'
with open(file_path, 'r', encoding='utf-8') as jf:
    exif_list = json.load(jf)


In [17]:
exif_data = exif_list[0]
jpeg_info = exif_data.get("Jpeg2000", {})

In [18]:
w = jpeg_info.get('ImageWidth')
h = jpeg_info.get('ImageHeight')

### Inspect content items of one issue

In [19]:
for ci in issue.content_items[:5]:
    print("CI ID:", ci["m"]["id"])
    print("Type:", ci["m"]["tp"])
    print("Pages:", ci["m"]["pp"])
    print("Legacy info:", ci.get("l"))
    print("IIIF link:", ci["m"].get("iiif_link"))
    print("-" * 50)

CI ID: DP-1987-10-15-a-i0001
Type: page
Pages: [1]
Legacy info: {'id': 'DP-1987-10-15-a-p0001', 'parts': [{'comp_role': 'page', 'comp_id': 'DP-1987-10-15-a-p0001', 'comp_fileid': 'DP_0879_1987_10_15_01_page_1.xml', 'comp_page_no': 1}], 'source': {'mit': 'DP_0879_1987_10_15_01_mit.xml', 'page_xml': ['DP_0879_1987_10_15_01_page_1.xml'], 'page_image': ['https://www.scriptorium.ch/api/iiif-img/v3/1782642']}}
IIIF link: None
--------------------------------------------------
CI ID: DP-1987-10-15-a-i0002
Type: page
Pages: [2]
Legacy info: {'id': 'DP-1987-10-15-a-p0002', 'parts': [{'comp_role': 'page', 'comp_id': 'DP-1987-10-15-a-p0002', 'comp_fileid': 'DP_0879_1987_10_15_01_page_2.xml', 'comp_page_no': 2}], 'source': {'mit': 'DP_0879_1987_10_15_01_mit.xml', 'page_xml': ['DP_0879_1987_10_15_01_page_2.xml'], 'page_image': ['https://www.scriptorium.ch/api/iiif-img/v3/1782643']}}
IIIF link: None
--------------------------------------------------
CI ID: DP-1987-10-15-a-i0003
Type: page
Pages: [3]

In [20]:
DPdir = dir2issue("/mnt/project_impresso/original/BCUL/Domaine_Public/1987/10/15/169220", alias_mapping["Domaine_Public"])

mit file ends with: /mnt/project_impresso/original/BCUL/Domaine_Public/1987/10/15/169220/DP_0879_1987_10_15_01_mit.xml xml both


In [21]:
# create issue object
from collections import Counter
import json

# --- 1. Load issue ---
issue = BculNewspaperIssue(DPdir)

# --- 2. Overview ---
types = [ci["m"]["tp"] for ci in issue.content_items]
print("🗞️  Counts by CI type:", Counter(types))
print(f"Total CIs: {len(issue.content_items)}\n")

# --- 3. Inspect first examples by type ---
def show_example(ci_type, max_show=2):
    cis = [ci for ci in issue.content_items if ci["m"]["tp"] == ci_type]
    if not cis:
        print(f"No content items of type '{ci_type}' found.\n")
        return
    print(f"🔍 Example {ci_type} CI ({len(cis)} total):\n")
    for ci in cis[:max_show]:
        print(json.dumps(ci["m"], indent=2))
        print("Legacy:")
        print(json.dumps(ci["l"], indent=2))
        if "c" in ci:
            print("Coordinates:", ci["c"])
        print("-" * 80)
    print()

show_example("page")
show_example("image")
show_example("table")

# --- 4. Sanity checks ---
print("✅ Running consistency checks...")

# Check that all required fields exist
for ci in issue.content_items:
    assert "m" in ci, "Missing 'm' section!"
    assert "l" in ci, f"Missing 'l' (legacy) section in CI {ci['m']['id']}"
    assert "id" in ci["m"], "Missing m.id!"
    assert ci["m"]["pp"], f"Missing page number (pp) in CI {ci['m']['id']}"

# Check that all CIs have a valid page legacy id
for ci in issue.content_items:
    lid = ci["l"].get("id")
    assert lid, f"Missing legacy id in CI {ci['m']['id']}"

# Check that all images/tables share the same legacy id as their page
page_legacy_ids = {
    ci["m"]["pp"][0]: ci["l"]["id"]
    for ci in issue.content_items if ci["m"]["tp"] == "page"
}
for ci in issue.content_items:
    # ✅ correct relationship check
    if ci["m"]["tp"] in {"image", "table"}:
        pnum = ci["m"]["pp"][0]
        # make sure it belongs to a valid page and comp_id exists
        assert "comp_id" in ci["l"]["parts"][0], f"Missing comp_id in {ci['m']['id']}"
        assert ci["l"]["parts"][0]["comp_page_no"] == pnum, f"CI {ci['m']['id']} has wrong page reference"

        
# Optional: check for missing comp_id
missing_comp_ids = [
    ci["m"]["id"] for ci in issue.content_items
    if not ci["l"]["parts"][0].get("comp_id")
]

# print which content items are missing comp_id
if missing_comp_ids:
    print(f"⚠️  {len(missing_comp_ids)} CIs have no comp_id in legacy parts:")
    for cid in missing_comp_ids:
        ci = next(c for c in issue.content_items if c["m"]["id"] == cid)
        print(f"  - CI ID: {cid} (id: {ci['l']['id']}, type: {ci['m']['tp']}, pages: {ci['m']['pp']})")
else:
    print("✅ All CIs have comp_id values.")

print("✅ All sanity checks passed successfully.\n")


# --- 5. Summary per page ---
print("Pages summary (page_no → CI types):")
from collections import defaultdict
page_summary = defaultdict(list)
for ci in issue.content_items:
    tp = ci["m"]["tp"]
    for p in ci["m"]["pp"]:
        page_summary[p].append(tp)
for p in sorted(page_summary):
    print(f"Page {p}: {Counter(page_summary[p])}")


🗞️  Counts by CI type: Counter({'page': 8, 'image': 7, 'table': 4})
Total CIs: 19

🔍 Example page CI (8 total):

{
  "id": "DP-1987-10-15-a-i0001",
  "pp": [
    1
  ],
  "tp": "page",
  "ro": 1
}
Legacy:
{
  "id": "DP-1987-10-15-a-p0001",
  "parts": [
    {
      "comp_role": "page",
      "comp_id": "DP-1987-10-15-a-p0001",
      "comp_fileid": "DP_0879_1987_10_15_01_page_1.xml",
      "comp_page_no": 1
    }
  ],
  "source": {
    "mit": "DP_0879_1987_10_15_01_mit.xml",
    "page_xml": [
      "DP_0879_1987_10_15_01_page_1.xml"
    ],
    "page_image": [
      "https://www.scriptorium.ch/api/iiif-img/v3/1782642"
    ]
  }
}
--------------------------------------------------------------------------------
{
  "id": "DP-1987-10-15-a-i0002",
  "pp": [
    2
  ],
  "tp": "page",
  "ro": 3
}
Legacy:
{
  "id": "DP-1987-10-15-a-p0002",
  "parts": [
    {
      "comp_role": "page",
      "comp_id": "DP-1987-10-15-a-p0002",
      "comp_fileid": "DP_0879_1987_10_15_01_page_2.xml",
      "comp_

In [33]:
DPdir = dir2issue("/mnt/project_impresso/original/BCUL/Domaine_Public/1987/10/15/169220", alias_mapping["Domaine_Public"])
issue = BculNewspaperIssue(DPdir)

# Test snippet: check if issue_id and page_id were correctly added

# Pick the first 3 content items for quick inspection
for ci in issue.content_items[:3]:
    legacy = ci.get("l", {})
    m = ci.get("m", {})
    print(f"📰 CI ID: {m.get('id')}  (type: {m.get('tp')})")
    print(f"   → Issue ID: {legacy.get('issue_id')}")
    print(f"   → Page ID: {legacy.get('page_id')}")
    print(f"   → Page number: {m.get('pp')}")
    print(f"   → Source XML: {legacy.get('source', {}).get('page_xml')}")
    print("-" * 80)

# Optional: sanity checks
issue_ids = {ci["l"].get("issue_id") for ci in issue.content_items}
page_ids = [ci["l"].get("page_id") for ci in issue.content_items if ci["l"].get("page_id")]

print(f"✅ Unique issue IDs found: {issue_ids}")
print(f"✅ Example of page IDs: {page_ids[:5]}")


mit file ends with: /mnt/project_impresso/original/BCUL/Domaine_Public/1987/10/15/169220/DP_0879_1987_10_15_01_mit.xml xml both
📰 CI ID: DP-1987-10-15-a-i0001  (type: page)
   → Issue ID: 169220
   → Page ID: 1782642
   → Page number: [1]
   → Source XML: ['DP_0879_1987_10_15_01_page_1.xml']
--------------------------------------------------------------------------------
📰 CI ID: DP-1987-10-15-a-i0002  (type: image)
   → Issue ID: 169220
   → Page ID: 1782642
   → Page number: [1]
   → Source XML: ['DP_0879_1987_10_15_01_page_1.xml']
--------------------------------------------------------------------------------
📰 CI ID: DP-1987-10-15-a-i0003  (type: page)
   → Issue ID: 169220
   → Page ID: 1782643
   → Page number: [2]
   → Source XML: ['DP_0879_1987_10_15_01_page_2.xml']
--------------------------------------------------------------------------------
✅ Unique issue IDs found: {169220}
✅ Example of page IDs: [1782642, 1782642, 1782643, 1782643, 1782644]


In [34]:
print(f"📰 Issue ID: {issue.id}")
print(f"Total content items: {len(issue.content_items)}\n")

for ci in issue.content_items:
    ci_id = ci.get("m", {}).get("id", "UNKNOWN_CI")
    l_id = ci.get("l", {}).get("file_id", "UNKNOWN_LEGACY_ID")
    ci_type = ci.get("m", {}).get("tp", "UNKNOWN_TYPE")
    issue_id = ci.get("l", {}).get("issue_id", "❌ MISSING")
    page_id = ci.get("l", {}).get("page_id", "❌ MISSING")
    parts = ci.get("l", {}).get("parts", [])
    comp_ids = [p.get("comp_id", "❌ MISSING") for p in parts]
    print(f"- CI ID: {ci_id} | File_id: {l_id} | Type: {ci_type} | issue_id: {issue_id} | page_id: {page_id} | comp_ids: {comp_ids}")

print("\n✅ Done listing all comp_ids for this issue.")


📰 Issue ID: DP-1987-10-15-a
Total content items: 19

- CI ID: DP-1987-10-15-a-i0001 | File_id: DP_0879_1987_10_15_01_page_1 | Type: page | issue_id: 169220 | page_id: 1782642 | comp_ids: [['{B88CB401-BAB8-4530-8659-27F79A700C8B}', '{BD734612-6B7F-4055-9116-3FEA0C8F077C}', '{4BC1705A-0314-4F8D-90F8-141CA0F808D8}', '{AD293151-9AA8-41AB-95D6-E86EA9D5306A}', '{28C8A244-6AB5-4B0B-9366-C2DB7B282EE6}']]
- CI ID: DP-1987-10-15-a-i0002 | File_id: DP_0879_1987_10_15_01_page_1 | Type: image | issue_id: 169220 | page_id: 1782642 | comp_ids: ['{91A2C083-D8FC-4AFC-9059-2038E51A72BC}']
- CI ID: DP-1987-10-15-a-i0003 | File_id: DP_0879_1987_10_15_01_page_2 | Type: page | issue_id: 169220 | page_id: 1782643 | comp_ids: [['{53951ED6-9AA6-4DB5-83D3-450FEE3A2CAD}', '{83FDCCAF-11B5-4A70-A299-D54E8C2AF58A}']]
- CI ID: DP-1987-10-15-a-i0004 | File_id: DP_0879_1987_10_15_01_page_2 | Type: table | issue_id: 169220 | page_id: 1782643 | comp_ids: ['{F93C713D-66EB-432A-BF96-37D2D3288F2C}']
- CI ID: DP-1987-10-15-

In [28]:
filename = "123456.xml"
page_id = os.path.splitext(filename)[0]

In [29]:
page_id

'123456'